# 25. 같이 변이되는 유전자 임베딩 · 유전자 그룹 비율 (H25)

## 배경 (11/11 회의 25번, "여유 있으면")
- 유전자 4,384개를 0/1로 따로 보면 드문 유전자는 신호가 약함 → **같이 잘 변이되는 유전자끼리 묶어서** 보면 정보가 모임
- 외부 데이터·사전 학습 임베딩은 쓰지 않고, **train(fold 학습 부분)의 환자 × 유전자 변이 행렬만으로** 유전자 임베딩을 만듦
- 22 C(LB 0.3650)가 오른 이유는 test 변이 2배에 덜 흔들리는 **비율** 피처였음 → 이번에도 환자 피처는 개수가 아니라 **평균·비율**로 만듦

## 방법
1. fold 학습 부분의 이진 행렬(환자 × 유전자, any)에 TruncatedSVD(32차원) → 유전자마다 32차원 벡터 (같이 변이되는 유전자는 가까운 벡터)
2. 환자 피처 두 가지 (모두 한 환자의 변이 유전자들로 계산, 변이 수가 2배여도 값이 비슷)
   - **25a 임베딩 평균**: 환자의 변이 유전자 벡터 평균 (32개 열)
   - **25b 유전자 그룹 비율**: 유전자 벡터를 KMeans로 30그룹 → 환자의 변이 유전자 중 각 그룹 비율 (30개 열)
3. SVD·KMeans는 **fold 학습 부분에서만 fit** (Pipeline 안), 검증·test는 transform만 → Data Leakage 아님

## 가설
- H25a: 22b에 임베딩 평균을 더하면 22b보다 오른다
- H25b: 22b에 유전자 그룹 비율을 더하면 22b보다 오른다
- 더 나은 쪽은 XGB 0.7 + LR 0.3 섞기와 시드 평균 5개로 C와 비교 (팀 규칙: 실험 결과물 + 시드 평균 결과물)

## 검증 방법
- 모델: 10 최적 설정 XGB 고정, 데이터: `load_data()` (정규화), 22b 피처(any + ratio + 치환 비율 상위 100)는 22와 같은 계산
- 비교 (같은 fold): 22b vs 25a, 22b vs 25b / 섞기는 C(22b XGB 0.7 + LR 0.3, LB 0.3650)와 비교
- 채택 기준: 비교 대상보다 5 fold 중 4개 이상 상승
- 실행 환경: macOS M1 (Windows와 XGB 점수 ±0.004 차이 가능), 제출 파일은 Windows에서만 저장

#### 준비: 데이터, 22b 피처, 10 최적 설정

In [1]:
import sys; sys.path.append('..')
import time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission, macro_f1

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

# 22b 피처 (22 노트북과 동일): 미스센스 치환 종류 비율, 상위 100개는 fold 학습 부분에서만
def sub_ratio_matrix(df):
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class TopNSubs(BaseEstimator, TransformerMixin):
    def __init__(self, n=100):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n])
        self.drop_ = [c for c in sub if c not in keep]
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)
X22b, X22b_te = pd.concat([F_tr['any'], F_tr['ratio'], S_tr], axis=1), pd.concat([F_te['any'], F_te['ratio'], S_te], axis=1)
print('22b 열 수:', X22b.shape[1])

Darwin | CPU arm64 | xgboost 3.4.1
22b 열 수: 4632


#### 유전자 임베딩 transformer (fold 학습 부분에서만 fit)
- `fit`: 학습 부분의 유전자 0/1 열로 TruncatedSVD(32) → 유전자 벡터, (그룹 모드면) KMeans(30)로 유전자 그룹
- `transform`: 환자의 변이 유전자 벡터 평균(`emb`) 또는 그룹별 비율(`grp`)을 열로 붙임, 변이 0개 환자는 0

In [2]:
class GeneEmbed(BaseEstimator, TransformerMixin):
    def __init__(self, mode='emb', k=32, n_groups=30):
        self.mode, self.k, self.n_groups = mode, k, n_groups
    def fit(self, X, y=None):
        B = X[genes].to_numpy(np.float32)
        svd = TruncatedSVD(n_components=self.k, random_state=SEED).fit(B)
        self.V_ = svd.components_.T.astype(np.float32)                 # 유전자 × k
        if self.mode == 'grp':
            Vn = self.V_ / np.maximum(np.linalg.norm(self.V_, axis=1, keepdims=True), 1e-8)
            self.group_ = KMeans(n_clusters=self.n_groups, n_init=10, random_state=SEED).fit_predict(Vn)
        return self
    def transform(self, X):
        B = X[genes].to_numpy(np.float32)
        n = np.maximum(B.sum(axis=1, keepdims=True), 1)
        if self.mode == 'emb':
            E = (B @ self.V_) / n
            cols = [f'gemb_{i}' for i in range(self.k)]
        else:
            G = np.zeros((len(genes), self.n_groups), np.float32)
            G[np.arange(len(genes)), self.group_] = 1
            E = (B @ G) / n
            cols = [f'ggrp_{i}' for i in range(self.n_groups)]
        return pd.concat([X, pd.DataFrame(E, index=X.index, columns=cols)], axis=1)

# 확인용 (train 전체로 한 번 fit, 점수 계산에는 안 씀): 같은 그룹에 묶인 유전자 예시
chk = GeneEmbed('grp').fit(X22b)
top = pd.Series(chk.group_, index=genes)
freq = F_tr['any'].sum()
for g in top.value_counts().index[:5]:
    members = freq[top[top == g].index].sort_values(ascending=False).head(6)
    print(f'그룹 {g} ({(top == g).sum()}개):', ', '.join(members.index))

그룹 9 (275개): TP53, PIK3CA, RYR2, PCLO, SPTA1, RELN
그룹 16 (196개): PREX1, SPTBN1, RET, COL17A1, MMP16, NRP2
그룹 27 (191개): DMD, APC, DOCK2, CACNA1A, TNC, MYOM2
그룹 13 (181개): PDHA2, RAPGEF6, GAD2, TGFBR2, ADAM23, ZBTB16
그룹 0 (177개): CDH8, CENPE, PLCL1, ABCA1, CR2, PTPRM


#### 비교: 22b / 25a / 25b
- 시간이 걸림 (XGB 3개, 약 12분)

In [3]:
MODELS = {
    '22b: any + ratio + 치환 100': lambda: make_pipeline(TopNSubs(100), BalancedXGB(**BEST)),
    '25a: 22b + 유전자 임베딩 평균': lambda: make_pipeline(GeneEmbed('emb'), TopNSubs(100), BalancedXGB(**BEST)),
    '25b: 22b + 유전자 그룹 비율': lambda: make_pipeline(GeneEmbed('grp'), TopNSubs(100), BalancedXGB(**BEST)),
}
R = {}
for name, mk in MODELS.items():
    t = time.time()
    R[name] = run_cv(mk, X22b, y, X22b_te, folds=folds, verbose=False)
    print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")

fs = lambda k: np.array(R[k]['fold_scores'])
base = '22b: any + ratio + 치환 100'
pd.DataFrame([{'피처': k, 'CV': round(fs(k).mean(), 4), 'std': round(fs(k).std(), 4), 'OOF': round(R[k]['oof_score'], 4),
               '22b 대비': round(fs(k).mean() - fs(base).mean(), 4) if k != base else '-',
               '22b보다 높은 fold': f'{int((fs(k) > fs(base)).sum())}/5' if k != base else '-'} for k in R])

22b: any + ratio + 치환 100: CV 0.4487 ± 0.0065 (OOF 0.4497, 226s)
25a: 22b + 유전자 임베딩 평균: CV 0.4315 ± 0.0070 (OOF 0.4324, 219s)
25b: 22b + 유전자 그룹 비율: CV 0.4501 ± 0.0101 (OOF 0.4504, 220s)


,피처,CV,std,OOF,22b 대비,22b보다 높은 fold
0,22b: any + ratio + 치환 100,0.4487,0.0065,0.4497,-,-
1,25a: 22b + 유전자 임베딩 평균,0.4315,0.0070,0.4324,-0.0172,0/5
2,25b: 22b + 유전자 그룹 비율,0.4501,0.0101,0.4504,0.0014,3/5


#### fold별 점수

In [4]:
pd.DataFrame({k: np.round(fs(k), 4) for k in R}, index=[f'fold {i}' for i in range(5)])

,22b: any + ratio + 치환 100,25a: 22b + 유전자 임베딩 평균,25b: 22b + 유전자 그룹 비율
fold 0,0.4540,0.4190,0.4363
fold 1,0.4426,0.4370,0.4624
fold 2,0.4407,0.4393,0.4500
fold 3,0.4485,0.4306,0.4418
fold 4,0.4578,0.4317,0.4600


#### 섞기 vs C (XGB 0.7 + LR 0.3, 같은 fold 짝 비교)
- 25a·25b 중 CV가 더 높은 쪽을 C(22b XGB 0.7 + LR 0.3, **LB 0.3650**)와 비교
- LR은 14·C와 같은 설정 (any, C=0.1, balanced), 비율 0.7 / 0.3도 같음

In [5]:
best25 = max([k for k in R if k.startswith('25')], key=lambda k: R[k]['cv_mean'])
print('섞기에 쓸 25 설정:', best25)
r_lr = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
              F_tr['any'].astype('float32'), y, F_te['any'].astype('float32'), folds=folds, verbose=False)
blend = lambda r: (0.7 * r['oof_proba'] + 0.3 * r_lr['oof_proba'], 0.7 * r['test_proba'] + 0.3 * r_lr['test_proba'])
fsb = lambda P: np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in folds])
coad, stes = list(le.classes_).index('COAD'), list(le.classes_).index('STES')
BL = {'C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)': blend(R[base]), f'{best25[:3]} XGB 0.7 + LR 0.3': blend(R[best25])}
bC = fsb(BL['C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)'][0])

def table(bl, refs):
    rows = []
    for name, (po, pt) in bl.items():
        s, pred = fsb(po), pt.argmax(1)
        row = {'조합': name, 'CV': round(s.mean(), 4), 'std': round(s.std(), 4), 'OOF': round(macro_f1(y, po.argmax(1)), 4)}
        for rname, ref in refs.items():
            row[f'{rname}보다 높은 fold'] = f'{int((s > ref).sum())}/5'
        row.update({'test COAD 예측': f'{np.mean(pred == coad):.1%}', 'test STES 예측': f'{np.mean(pred == stes):.1%}'})
        rows.append(row)
    return pd.DataFrame(rows)

print('fold별:', {k: np.round(fsb(v[0]), 4).tolist() for k, v in BL.items()})
table(BL, {'C': bC})

섞기에 쓸 25 설정: 25b: 22b + 유전자 그룹 비율
fold별: {'C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)': [0.4594, 0.4437, 0.4564, 0.456, 0.4628], '25b XGB 0.7 + LR 0.3': [0.4374, 0.4602, 0.4526, 0.4494, 0.4648]}


,조합,CV,std,OOF,C보다 높은 fold,test COAD 예측,test STES 예측
0,C: 22b XGB 0.7 + LR 0.3 (LB 0.3650),0.4557,0.0065,0.4565,0/5,15.0%,20.1%
1,25b XGB 0.7 + LR 0.3,0.4529,0.0095,0.4531,2/5,15.2%,20.2%


#### 시드 평균 5개 (팀 규칙: 실험 결과물 + 시드 평균 결과물)
- 17과 같은 방식: XGB만 시드 5개(42, 0, 1, 2, 3)로 학습해 확률 평균, fold(seed 42)와 LR·비율 0.7 / 0.3은 고정
- SVD·KMeans는 seed 42로 고정 (XGB 시드만 바꿈)
- 시간이 오래 걸림 (XGB 8번 추가, 약 30분)

In [6]:
SEEDS = [42, 0, 1, 2, 3]

def xgb_seed_avg(name):
    oofs, tests = [R[name]['oof_proba']], [R[name]['test_proba']]   # 시드 42 = 위에서 돌린 결과
    steps = MODELS[name]().steps[:-1]
    for s in SEEDS[1:]:
        t = time.time()
        params = {**BEST, 'random_state': s}
        r = run_cv(lambda: make_pipeline(*[type(st)(**st.get_params()) for _, st in steps], BalancedXGB(**params)),
                   X22b, y, X22b_te, folds=folds, verbose=False)
        oofs.append(r['oof_proba']); tests.append(r['test_proba'])
        print(f"{name} 시드 {s}: CV {r['cv_mean']:.4f} ({time.time() - t:.0f}s)")
    return {'oof_proba': np.mean(oofs, axis=0), 'test_proba': np.mean(tests, axis=0)}

BL_S = {'C 시드 평균': blend(xgb_seed_avg(base)), f'{best25[:3]} + LR 시드 평균': blend(xgb_seed_avg(best25))}
bCs = fsb(BL_S['C 시드 평균'][0])
table({**BL, **BL_S}, {'C(시드 42)': bC, 'C 시드 평균': bCs})

22b: any + ratio + 치환 100 시드 0: CV 0.4507 (234s)
22b: any + ratio + 치환 100 시드 1: CV 0.4566 (198s)
22b: any + ratio + 치환 100 시드 2: CV 0.4534 (220s)
22b: any + ratio + 치환 100 시드 3: CV 0.4546 (218s)
25b: 22b + 유전자 그룹 비율 시드 0: CV 0.4431 (235s)
25b: 22b + 유전자 그룹 비율 시드 1: CV 0.4494 (231s)
25b: 22b + 유전자 그룹 비율 시드 2: CV 0.4491 (223s)
25b: 22b + 유전자 그룹 비율 시드 3: CV 0.4416 (223s)


,조합,CV,std,OOF,C(시드 42)보다 높은 fold,C 시드 평균보다 높은 fold,test COAD 예측,test STES 예측
0,C: 22b XGB 0.7 + LR 0.3 (LB 0.3650),0.4557,0.0065,0.4565,0/5,1/5,15.0%,20.1%
1,25b XGB 0.7 + LR 0.3,0.4529,0.0095,0.4531,2/5,1/5,15.2%,20.2%
2,C 시드 평균,0.4622,0.0082,0.4631,4/5,0/5,15.0%,20.0%
3,25b + LR 시드 평균,0.4583,0.0069,0.4589,3/5,2/5,14.1%,21.2%


#### 제출 파일 (데스크탑에서만 저장)
- 결과물 2개 (시드 42판 + 시드 평균판), 채택 여부는 C와의 fold 비교로 판단

In [7]:
tag = best25[:3]
OUTS = {f'blend_gene{tag}_0.7_LR0.3': (BL[f'{tag} XGB 0.7 + LR 0.3'], bC),
        f'blend_gene{tag}_0.7_LR0.3_seed5': (BL_S[f'{tag} + LR 시드 평균'], bCs)}
for fname, ((po, pt), ref) in OUTS.items():
    up = int((fsb(po) > ref).sum())
    print(f"{fname}: 비교 대상(C)보다 높은 fold {up}/5 → {'채택 기준 통과' if up >= 4 else '기준 미달'}")
    if platform.system() == 'Windows':
        save_submission(submission, pt, le, name=fname, cv_score=macro_f1(y, po.argmax(1)))   # 파일명 숫자 = OOF
if platform.system() != 'Windows':
    print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

blend_gene25b_0.7_LR0.3: 비교 대상(C)보다 높은 fold 2/5 → 기준 미달
blend_gene25b_0.7_LR0.3_seed5: 비교 대상(C)보다 높은 fold 2/5 → 기준 미달
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


## 결과 (macOS M1)

**유전자 그룹 확인 (train 전체로 한 번 fit, 점수 계산에는 안 씀)**: train 데이터만으로 같이 변이되는 유전자가 묶임
- 예) 그룹 9: TP53, PIK3CA, RYR2 … / 그룹 27: DMD, APC, DOCK2 … / 그룹 16: PREX1, SPTBN1, RET …

**22b 대비 (같은 fold, XGB 단독)**

| 피처 | CV (± std) | OOF | 22b 대비 | 22b보다 높은 fold |
| :--- | :-: | :-: | :-: | :-: |
| 22b: any + ratio + 치환 100 | 0.4487 ± 0.0065 | 0.4497 | — | — |
| 25a: 22b + 유전자 임베딩 평균 (32열) | 0.4315 ± 0.0070 | 0.4324 | −0.0172 | 0/5 |
| 25b: 22b + 유전자 그룹 비율 (30열) | 0.4501 ± 0.0101 | 0.4504 | +0.0014 | 3/5 |

- 25a(임베딩 평균)는 5 fold 모두 하락 → 32차원 평균 벡터가 개별 유전자 신호를 흐림 (평균하면 환자마다 비슷한 값이 됨)
- 25b(그룹 비율)는 22b와 거의 같음 (+0.0014, 3/5, fold 0은 −0.018) → 채택 기준 미달

**섞기 vs C, 시드 평균 5개 (25b 사용, 같은 fold 짝 비교)**

| 조합 | CV (± std) | OOF | C(시드 42)보다 높은 fold | C 시드 평균보다 높은 fold | test COAD | test STES |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: |
| C: 22b XGB 0.7 + LR 0.3 (LB 0.3650) | 0.4557 ± 0.0065 | 0.4565 | — | 1/5 | 15.0% | 20.1% |
| 25b XGB 0.7 + LR 0.3 | 0.4529 ± 0.0095 | 0.4531 | 2/5 (−0.0028) | 1/5 | 15.2% | 20.2% |
| C 시드 평균 (XGB 시드 5개) | 0.4622 ± 0.0082 | 0.4631 | 4/5 | — | 15.0% | 20.0% |
| 25b + LR 시드 평균 | 0.4583 ± 0.0069 | 0.4589 | 3/5 | 2/5 (−0.0039) | 14.1% | 21.2% |

- 25b + LR은 C보다 2/5 fold, 시드 평균끼리도 2/5 fold → 둘 다 기준 미달
- test 예측 분포는 C와 거의 같음 → 유전자 그룹 비율이 22b + LR에 새 정보를 더하지 못함
- 시드별 CV: 22b 0.4487~0.4566, 25b 0.4416~0.4501 (25b가 시드에 따라 더 흔들림)

## 결론
- **H25a 기각**: 유전자 임베딩 평균은 22b 대비 −0.017, 0/5 fold
- **H25b 기각**: 유전자 그룹 비율은 22b 대비 +0.0014, 3/5 fold → 채택 기준(4/5) 미달
- **섞기·시드 평균도 기각**: 25b + LR은 C보다 2/5, 시드 평균끼리도 2/5 fold
- 같이 변이되는 유전자 정보는 22b(유전자 0/1 + 비율 + 치환 비율)에 이미 대부분 들어 있는 것으로 보임 → 제출 후보는 C 계열 유지
- 외부 데이터·사전 학습 임베딩은 쓰지 않았고, SVD·KMeans는 fold 학습 부분에서만 fit (Data Leakage 아님)
- 결과물 (데스크탑에서 실행하면 저장, 기준 미달이라 기록용): `blend_gene25b_0.7_LR0.3`, `blend_gene25b_0.7_LR0.3_seed5`
